In [1]:
import pandas as pd
import numpy as np
import random
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


### Upload CSV Files

Please upload your two CSV files using the file uploader below.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving MDA EXP-1 csv file 1 Areeba Khanam _02101182023.csv to MDA EXP-1 csv file 1 Areeba Khanam _02101182023.csv
Saving MDA EXP-1 csv file 2 Areeba Khanam _02101182023.csv to MDA EXP-1 csv file 2 Areeba Khanam _02101182023.csv


In [3]:
print(uploaded.keys())

dict_keys(['MDA EXP-1 csv file 1 Areeba Khanam _02101182023.csv', 'MDA EXP-1 csv file 2 Areeba Khanam _02101182023.csv'])


In [4]:
df1 = pd.read_csv("MDA EXP-1 csv file 1 Areeba Khanam _02101182023.csv")
print("CSV 1 (1 Entry) Content:")
print(df1)

df2 = pd.read_csv("MDA EXP-1 csv file 2 Areeba Khanam _02101182023.csv")
print("\nCSV 2 (5 Entries) Content:")
print(df2)

CSV 1 (1 Entry) Content:
   ReviewID                                         ReviewText
0         1  Customers expect a smooth and easy shopping ex...

CSV 2 (5 Entries) Content:
   ReviewID                                         ReviewText
0         1  Customers expect a smooth and easy shopping ex...
1         2  Quick and helpful customer support improves th...
2         3  A positive experience encourages customers to ...
3         4  I am feeling very happy with the product quali...
4         5  the service was fine , need little more improv...


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

def tokenize_text(text):
    return word_tokenize(text)

def remove_stopwords_fn(tokens):
    return [w for w in tokens if w not in stop_words]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

In [6]:
def process_dataframe(df, name):
    print(f"\n================ Processing {name} ================\n")
    tokens_list, nostop_list, lemma_list = [], [], []

    for _, row in df.iterrows():
        text = row['ReviewText']
        print(f"\n--- ReviewID {row['ReviewID']} ---")
        print("Original Text:", text)

        cleaned = clean_text(text)
        tokens = tokenize_text(cleaned)
        print("Tokens:", tokens)

        nostop = remove_stopwords_fn(tokens)
        print("After Stopword Removal:", nostop)

        lemma = lemmatize_tokens(nostop)
        print("After Lemmatization:", lemma)

        tokens_list.append(tokens)
        nostop_list.append(nostop)
        lemma_list.append(lemma)

    df['Tokens'] = tokens_list
    df['NoStopwords'] = nostop_list
    df['Lemmatized'] = lemma_list
    df['Lemmatized_Text'] = df['Lemmatized'].apply(lambda x: " ".join(x))
    return df

df1 = process_dataframe(df1, "CSV 1 (1 Entry)")
df2 = process_dataframe(df2, "CSV 2 (5 Entries)")


================ Processing CSV 1 (1 Entry) ================


--- ReviewID 1 ---
Original Text: Customers expect a smooth and easy shopping experience.
Tokens: ['customers', 'expect', 'a', 'smooth', 'and', 'easy', 'shopping', 'experience']
After Stopword Removal: ['customers', 'expect', 'smooth', 'easy', 'shopping', 'experience']
After Lemmatization: ['customer', 'expect', 'smooth', 'easy', 'shopping', 'experience']

================ Processing CSV 2 (5 Entries) ================


--- ReviewID 1 ---
Original Text: Customers expect a smooth and easy shopping experience.
Tokens: ['customers', 'expect', 'a', 'smooth', 'and', 'easy', 'shopping', 'experience']
After Stopword Removal: ['customers', 'expect', 'smooth', 'easy', 'shopping', 'experience']
After Lemmatization: ['customer', 'expect', 'smooth', 'easy', 'shopping', 'experience']

--- ReviewID 2 ---
Original Text: Quick and helpful customer support improves their overall experience.
Tokens: ['quick', 'and', 'helpful', 'customer', '

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_step_by_step(df, name):
    print(f"\n================ TF-IDF for {name} ================\n")
    corpus = df['Lemmatized_Text'].tolist()

    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(corpus)
    feature_names = tfidf.get_feature_names_out()

    print("Total Features (unique words):", len(feature_names))
    print("\nFeatures one-by-one:")
    for i, f in enumerate(feature_names):
        print(f"{i+1}. {f}")

    print("\nFull TF-IDF Matrix:")
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=df['ReviewID'])
    print(tfidf_df)
    return tfidf, tfidf_matrix, tfidf_df

tfidf1, matrix1, tfidf_df1 = tfidf_step_by_step(df1, "CSV 1 (1 Entry)")
tfidf2, matrix2, tfidf_df2 = tfidf_step_by_step(df2, "CSV 2 (5 Entries)")


================ TF-IDF for CSV 1 (1 Entry) ================

Total Features (unique words): 6

Features one-by-one:
1. customer
2. easy
3. expect
4. experience
5. shopping
6. smooth

Full TF-IDF Matrix:
          customer      easy    expect  experience  shopping    smooth
ReviewID                                                              
1         0.408248  0.408248  0.408248    0.408248  0.408248  0.408248

================ TF-IDF for CSV 2 (5 Entries) ================

Total Features (unique words): 30

Features one-by-one:
1. back
2. color
3. come
4. customer
5. easy
6. encourages
7. expect
8. experience
9. feeling
10. fine
11. happy
12. helpful
13. improvement
14. improves
15. little
16. need
17. option
18. overall
19. positive
20. product
21. provide
22. quality
23. quick
24. service
25. shop
26. shopping
27. size
28. smooth
29. staff
30. support

Full TF-IDF Matrix:
              back     color      come  customer      easy  encourages  \
ReviewID                          

In [8]:
def stem_then_lemmatize(df, name):
    print(f"\n================ Extra Step (Stem → Lemmatize) for {name} ================\n")
    stemmed_col, stem_lemma_col = [], []

    for _, row in df.iterrows():
        lemma_tokens = row['Lemmatized']
        print(f"\n--- ReviewID {row['ReviewID']} ---")
        print("Input (Lemmatized Tokens):", lemma_tokens)

        stemmed = stem_tokens(lemma_tokens)
        print("After Stemming:", stemmed)

        final = lemmatize_tokens(stemmed)
        print("After Lemmatization (on stemmed output):", final)

        stemmed_col.append(stemmed)
        stem_lemma_col.append(final)

    df['Stemmed'] = stemmed_col
    df['Stemmed_then_Lemmatized'] = stem_lemma_col
    return df

df1 = stem_then_lemmatize(df1, "CSV 1 (1 Entry)")
df2 = stem_then_lemmatize(df2, "CSV 2 (5 Entries)")


================ Extra Step (Stem → Lemmatize) for CSV 1 (1 Entry) ================


--- ReviewID 1 ---
Input (Lemmatized Tokens): ['customer', 'expect', 'smooth', 'easy', 'shopping', 'experience']
After Stemming: ['custom', 'expect', 'smooth', 'easi', 'shop', 'experi']
After Lemmatization (on stemmed output): ['custom', 'expect', 'smooth', 'easi', 'shop', 'experi']

================ Extra Step (Stem → Lemmatize) for CSV 2 (5 Entries) ================


--- ReviewID 1 ---
Input (Lemmatized Tokens): ['customer', 'expect', 'smooth', 'easy', 'shopping', 'experience']
After Stemming: ['custom', 'expect', 'smooth', 'easi', 'shop', 'experi']
After Lemmatization (on stemmed output): ['custom', 'expect', 'smooth', 'easi', 'shop', 'experi']

--- ReviewID 2 ---
Input (Lemmatized Tokens): ['quick', 'helpful', 'customer', 'support', 'improves', 'overall', 'experience']
After Stemming: ['quick', 'help', 'custom', 'support', 'improv', 'overal', 'experi']
After Lemmatization (on stemmed output): ['